# LoRA fine-tuning Qwen2.5-0.5B-Instruct for financial tweet sentiment

This notebook runs the whole experiment end to end:

1. Look at the data (`zeroshot/twitter-financial-news-sentiment`, labels: Bearish / Bullish / Neutral)
2. Show the prompt format (Qwen's own chat template, one-word answer)
3. Evaluate the **base model** on the held-out test split
4. Train a **LoRA adapter with r=8** and evaluate it
5. Train a **LoRA adapter with r=16** and evaluate it
6. Compare the results and look at the errors

The training / evaluation logic lives in `train.py`, `evaluate_model.py` and `common.py`; this notebook only calls those functions, so the command-line scripts and the notebook always run the same code.

## 0. Setup

Locally: activate the conda env from the README and select it as the notebook kernel.
On Google Colab: choose a GPU runtime; the cell below installs the pinned libraries and clones the repo (Colab already ships PyTorch).

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q transformers==5.20.2 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0
    !git clone -q https://github.com/Takum1ouo/qwen-lora-finetune.git
    %cd qwen-lora-finetune

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")  # harmless Windows cache warning

'1'

In [2]:
import torch

print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported(), "(fp16 is used otherwise, e.g. on a T4)")

torch 2.14.1+cu126
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
bf16 supported: True (fp16 is used otherwise, e.g. on a T4)


## 1. Data

The dataset has a `train` split (9,543 tweets) and a `validation` split (2,388 tweets) but no test split.
The `validation` split is **held out from training** and serves as the test set.

The classes are imbalanced: about 65% of tweets are Neutral, so always answering "Neutral" already scores ~65.6% accuracy. That majority-class baseline is the number to beat, and it is why macro-F1 is reported next to accuracy.

In [3]:
import pandas as pd
from datasets import load_dataset

from common import DATASET_NAME, LABELS, TEST_SPLIT

ds = load_dataset(DATASET_NAME)
dist = pd.DataFrame({
    split: pd.Series(ds[split]["label"]).map(dict(enumerate(LABELS))).value_counts()
    for split in ("train", TEST_SPLIT)
})
dist["train %"] = (dist["train"] / dist["train"].sum() * 100).round(1)
dist[f"{TEST_SPLIT} %"] = (dist[TEST_SPLIT] / dist[TEST_SPLIT].sum() * 100).round(1)
dist

,train,validation,train %,validation %
Neutral,6178,1566,64.7,65.6
Bullish,1923,475,20.2,19.9
Bearish,1442,347,15.1,14.5


In [4]:
train_df = ds["train"].to_pandas()
for label_id, name in enumerate(LABELS):
    print(f"--- {name} ---")
    for text in train_df[train_df.label == label_id].sample(3, random_state=0).text:
        print("  ", text)

overlap = set(ds[TEST_SPLIT]["text"]) & set(ds["train"]["text"])
print(f"\ntest tweets that also appear in train: {len(overlap)}")

--- Bearish ---
   VF Corp. shares slide 1.9% premarket
   GBP/JPY Price Forecast – British Pound Fails To Break Out
   Why 51job Shares Dropped 15% Last Month
--- Bullish ---
   Intercontinental Exchange Q4 results in-line, data revenue +4%
   Gold steadies as virus concerns stall stock markets - Nasdaq
   Sanofi Sees Profit Rising in 2020
--- Neutral ---
   WORK, XPO, PYX and AMKR among after hour movers
   Eskom Loses Bid for Higher Tariffs as Court Defers to Regulator
   Edited Transcript of MMS earnings conference call or presentation 19-Nov-19 2:00pm GMT

test tweets that also appear in train: 0


## 2. Prompt format

Each tweet is wrapped in Qwen's chat template. During training only the answer tokens (the label word and the `<|im_end|>` end marker) carry a loss; all prompt tokens get label `-100`.

In [5]:
from transformers import AutoTokenizer

from common import MODEL_NAME, build_prompt

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
example = ds["train"][0]
print(build_prompt(tokenizer, example["text"]) + LABELS[example["label"]] + tokenizer.eos_token)
for label in LABELS:
    ids = tokenizer(label + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    print(f"{label:8} -> trained tokens: {[tokenizer.decode([i]) for i in ids]}")

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Classify the sentiment of the following financial tweet as Bearish, Bullish, or Neutral. Answer with exactly one word.

Tweet: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT<|im_end|>
<|im_start|>assistant
Bearish<|im_end|>
Bearish  -> trained tokens: ['Bear', 'ish', '<|im_end|>']
Bullish  -> trained tokens: ['B', 'ull', 'ish', '<|im_end|>']
Neutral  -> trained tokens: ['Neutral', '<|im_end|>']


## 3. Base model (before fine-tuning)

Greedy decoding, at most 10 new tokens. An output counts as *strictly formatted* if it is exactly one label word; accuracy also accepts a longer answer that mentions exactly one label.

In [6]:
from evaluate_model import evaluate_model

SUMMARY_KEYS = ["accuracy", "macro_f1", "format_rate", "unparseable_rate", "majority_baseline", "eval_seconds"]

results = {"base": evaluate_model("base")}
print({k: round(results["base"][k], 4) for k in SUMMARY_KEYS})
pd.DataFrame(results["base"]["confusion"]).T.fillna(0).astype(int)  # rows: gold, columns: predicted

{'accuracy': 0.3961, 'macro_f1': 0.2835, 'format_rate': 1.0, 'unparseable_rate': 0.0, 'majority_baseline': 0.6558, 'eval_seconds': 16.2}


,Bullish,Neutral
Bearish,319,28
Bullish,374,101
Neutral,994,572


## 4. LoRA r=8

LoRA adapters on `q_proj`, `k_proj`, `v_proj` of all 24 layers, `lora_alpha = 2 * r`, dropout 0.05, learning rate 2e-4 (linear decay), effective batch 16 (4 x 4 gradient accumulation), 1 epoch.

In [7]:
from train import train_lora

train_summaries = {"lora_r8": train_lora("outputs/lora_r8", r=8, alpha=16)}
print({k: v for k, v in train_summaries["lora_r8"].items() if k != "log_history"})

trainable params: 737,280 || all params: 494,770,048 || trainable%: 0.1490


Step,Training Loss
20,0.339489
40,0.231485
60,0.272730
80,0.219421
100,0.195014
120,0.172628
140,0.192635
160,0.162055
180,0.144723
200,0.183604


{'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'learning_rate': 0.0002, 'epochs': 1, 'batch_size': 4, 'grad_accum': 4, 'train_examples': 9543, 'trainable_params': 737280, 'train_seconds': 499.2, 'peak_gpu_gb': 2.65}


In [8]:
results["lora_r8"] = evaluate_model("lora_r8", adapter="outputs/lora_r8/adapter")
print({k: round(results["lora_r8"][k], 4) for k in SUMMARY_KEYS})
pd.DataFrame(results["lora_r8"]["confusion"]).T.fillna(0).astype(int)

{'accuracy': 0.8819, 'macro_f1': 0.848, 'format_rate': 1.0, 'unparseable_rate': 0.0, 'majority_baseline': 0.6558, 'eval_seconds': 15.9}


,Neutral,Bearish,Bullish
Bearish,65,276,6
Bullish,88,8,379
Neutral,1451,55,60


## 5. LoRA r=16

Same settings, only the rank changes. `lora_alpha` is doubled to 32 so the LoRA scaling `alpha / r` stays at 2.

In [9]:
train_summaries["lora_r16"] = train_lora("outputs/lora_r16", r=16, alpha=32)
print({k: v for k, v in train_summaries["lora_r16"].items() if k != "log_history"})

trainable params: 1,474,560 || all params: 495,507,328 || trainable%: 0.2976


Step,Training Loss
20,0.316722
40,0.203028
60,0.258402
80,0.205535
100,0.201639
120,0.171415
140,0.169920
160,0.150115
180,0.145868
200,0.172668


{'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'learning_rate': 0.0002, 'epochs': 1, 'batch_size': 4, 'grad_accum': 4, 'train_examples': 9543, 'trainable_params': 1474560, 'train_seconds': 1372.8, 'peak_gpu_gb': 2.66}


In [10]:
results["lora_r16"] = evaluate_model("lora_r16", adapter="outputs/lora_r16/adapter")
print({k: round(results["lora_r16"][k], 4) for k in SUMMARY_KEYS})
pd.DataFrame(results["lora_r16"]["confusion"]).T.fillna(0).astype(int)

{'accuracy': 0.8911, 'macro_f1': 0.859, 'format_rate': 1.0, 'unparseable_rate': 0.0, 'majority_baseline': 0.6558, 'eval_seconds': 15.8}


,Neutral,Bearish,Bullish
Bearish,65,277,5
Bullish,77,9,389
Neutral,1462,48,56


## 6. Comparison

In [11]:
rows = [{"model": "Majority class (always Neutral)", "accuracy": results["base"]["majority_baseline"]}]
for name, m in results.items():
    row = {"model": name, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "format_rate": m["format_rate"]}
    row.update({f"F1 {label}": m["per_class"][label]["f1"] for label in LABELS})
    if name in train_summaries:
        s = train_summaries[name]
        row.update({"trainable_params": s["trainable_params"], "train_min": s["train_seconds"] / 60,
                    "peak_gpu_gb": s["peak_gpu_gb"]})
    rows.append(row)
pd.DataFrame(rows).set_index("model").round(3)

,accuracy,macro_f1,format_rate,F1 Bearish,F1 Bullish,F1 Neutral,trainable_params,train_min,peak_gpu_gb
model,,,,,,,,,
Majority class (always Neutral),0.656,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
base,0.396,0.284,1.0,0.000,0.346,0.505,NaN,NaN,NaN
lora_r8,0.882,0.848,1.0,0.805,0.824,0.915,737280.0,8.32,2.65
lora_r16,0.891,0.859,1.0,0.814,0.841,0.922,1474560.0,22.88,2.66


Is the r=16 vs r=8 gap more than test-set noise? McNemar's exact test only looks at the tweets where the two models disagree. (This does not cover training randomness: each setting was trained once, with seed 42.)

In [12]:
from math import comb

from evaluate_model import load_predictions

p8, p16 = load_predictions("lora_r8"), load_predictions("lora_r16")
only_r8 = sum(a["pred"] == a["gold"] and b["pred"] != b["gold"] for a, b in zip(p8, p16))
only_r16 = sum(a["pred"] != a["gold"] and b["pred"] == b["gold"] for a, b in zip(p8, p16))
n, k = only_r8 + only_r16, min(only_r8, only_r16)
p_value = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2**n)
print(f"only r=8 correct: {only_r8}, only r=16 correct: {only_r16}, McNemar exact p = {p_value:.2g}")

only r=8 correct: 22, only r=16 correct: 44, McNemar exact p = 0.0092


In [13]:
def losses(name):
    return {h["step"]: h["loss"] for h in train_summaries[name]["log_history"] if "loss" in h}

loss_table = pd.DataFrame({"r=8": losses("lora_r8"), "r=16": losses("lora_r16")})
loss_table.iloc[::5].round(3)  # training loss every 100 optimizer steps

,r=8,r=16
20,0.339,0.317
120,0.173,0.171
220,0.132,0.127
320,0.177,0.162
420,0.184,0.173
520,0.155,0.148


## 7. Error analysis (r=16)

In [14]:
errors = pd.DataFrame([p for p in p16 if p["pred"] != p["gold"]])
print(f"{len(errors)} errors out of {len(p16)}")
print(errors.groupby(["gold", "pred"]).size().sort_values(ascending=False).to_string())
pd.set_option("display.max_colwidth", 120)
errors.sample(10, random_state=0)[["text", "gold", "pred"]]

260 errors out of 2388
gold     pred   
Bullish  Neutral    77
Bearish  Neutral    65
Neutral  Bullish    56
         Bearish    48
Bullish  Bearish     9
Bearish  Bullish     5


,text,gold,pred
233,"$GOOG $GOOGL - Alphabet: Flying Right To $1,700. Read more and get updates on any stock! https://t.co/cdftwzEnYJ #ec...",Neutral,Bullish
106,Tech War Set To Deepen As China Stockpiles US Chips Amid 'Silicon Curtain' Threat https://t.co/sMw7Z0Xo3w,Bearish,Neutral
237,"$NVDA $UBER - Beware The Ides Of December, Current Events Should Boost Markets Now; Sell Uber And Tesla. Get more u…...",Neutral,Bearish
76,Bipartisan Group Of US Lawmakers Introduce Bill Providing $3B To Buy Oil To Fill Strategic Petroleum Reserve $SPY… h...,Bullish,Neutral
173,Zimbabwe to the IMF: “Our fundamentals are now sound” https://t.co/HSSo4iqk7M,Bullish,Neutral
170,Illinois's Legalized Pot Sales Debut Beats All States Except One,Bullish,Neutral
8,Highlight: Edward Jones Sr. Equity Analyst Jennifer Roland on ExxonMobil reducing 2020 Capex by 30%: “Definitely a… ...,Bullish,Neutral
71,Empty LNG tankers are idling off Qatar as China slows imports amid the virus outbreak https://t.co/Vhf6L9WRYU,Bearish,Neutral
37,Sales of grounded Boeing 737 MAX gather pace at Dubai Airshow,Bullish,Neutral
152,Gm wins key legal victory over pre-bankruptcy liability,Bullish,Neutral
